# Notebook 01 - Pré-processamento de Áudio

In [1]:
import os
import librosa
import soundfile as sf

In [2]:
# Diretórios de entrada e saída
input_dir_a = '../../data/Hall-Reverb/Hall-Reverb'  # Entrada para o modelo
input_dir_b = '../../data/Chorus/Chorus'  # Target (saída desejada)
output_dir_a = '../../dataset/preprocessed_audio/Hall-Reverb/'
output_dir_b = '../../dataset/preprocessed_audio/Chorus/'

os.makedirs(output_dir_a, exist_ok=True)
os.makedirs(output_dir_b, exist_ok=True)

In [3]:
# Parâmetros
sample_rate = 22050  # Taxa de amostragem padrão
max_duration = 5.0  # Duração máxima (segundos)

In [4]:
# Função para normalizar e salvar áudio
def process_audio(input_path, output_path, sample_rate, max_duration):
    # Carregar áudio
    audio, sr = librosa.load(input_path, sr=sample_rate)
    
    # Normalizar duração
    if len(audio) > int(max_duration * sr):
        audio = audio[:int(max_duration * sr)]
    elif len(audio) < int(max_duration * sr):
        audio = librosa.util.fix_length(audio, int(max_duration * sr))
    
    # Salvar áudio processado
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    sf.write(output_path, audio, sr)


In [5]:
# Função para processar recursivamente os diretórios
def process_directories(input_dir_a, input_dir_b, output_dir_a, output_dir_b):
    for root_a, _, files_a in os.walk(input_dir_a):
        # Determinar o caminho relativo e correspondente para input_dir_b
        relative_path = os.path.relpath(root_a, input_dir_a)
        corresponding_dir_b = os.path.join(input_dir_b, relative_path)
        
        # Saídas correspondentes
        output_subdir_a = os.path.join(output_dir_a, relative_path)
        output_subdir_b = os.path.join(output_dir_b, relative_path)

        # Garantir que o diretório correspondente exista
        if not os.path.exists(corresponding_dir_b):
            print(f"Subdiretório correspondente não encontrado: {corresponding_dir_b}")
            continue

        # Ordenar os arquivos para garantir o pareamento
        files_a = sorted(f for f in files_a if f.endswith('.wav'))
        files_b = sorted(f for f in os.listdir(corresponding_dir_b) if f.endswith('.wav'))

        if len(files_a) != len(files_b):
            print(f"Número de arquivos diferente em {relative_path}. Pulando esta pasta.")
            continue

        # Processar cada par de arquivos
        for file_a, file_b in zip(files_a, files_b):
            input_path_a = os.path.join(root_a, file_a)
            input_path_b = os.path.join(corresponding_dir_b, file_b)
            
            output_path_a = os.path.join(output_subdir_a, file_a)
            output_path_b = os.path.join(output_subdir_b, file_b)
            
            process_audio(input_path_a, output_path_a, sample_rate, max_duration)
            process_audio(input_path_b, output_path_b, sample_rate, max_duration)

In [6]:
# Executar processamento
process_directories(input_dir_a, input_dir_b, output_dir_a, output_dir_b)

print("Pré-processamento concluído! Áudios processados e salvos.")

Pré-processamento concluído! Áudios processados e salvos.
